In [38]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor

In [39]:
df=pd.read_csv('diamonds.csv')


In [40]:
df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


In [41]:
df['depth'].unique()

array([61.5, 59.8, 56.9, 62.4, 63.3, 62.8, 62.3, 61.9, 65.1, 59.4, 64. ,
       60.4, 62.2, 60.2, 60.9, 62. , 63.4, 63.8, 62.7, 61. , 58.1, 62.5,
       60.5, 60. , 60.7, 59.5, 58.2, 64.1, 60.8, 61.8, 61.2, 61.1, 65.2,
       58.4, 63.1, 61.6, 59.3, 62.6, 63. , 63.2, 62.1, 61.4, 62.9, 63.7,
       59.2, 59.9, 57.9, 55.1, 57.5, 66.3, 61.7, 58.8, 64.5, 65.3, 59.6,
       64.4, 65.7, 63.6, 61.3, 60.1, 60.3, 58. , 64.6, 59.7, 57.8, 67.9,
       60.6, 57.2, 64.2, 65.8, 67.4, 59. , 63.5, 67.3, 58.7, 66.4, 68.1,
       63.9, 55. , 58.6, 64.3, 58.5, 65. , 56. , 58.3, 53.1, 64.9, 59.1,
       58.9, 66.7, 57.7, 65.4, 53.3, 53. , 67.8, 66.1, 55.8, 67.6, 68.2,
       65.5, 67.7, 69.5, 56.6, 56.3, 66.9, 66. , 67. , 57.6, 67.1, 65.6,
       64.8, 69.3, 66.2, 55.4, 66.8, 64.7, 66.6, 55.9, 57.3, 57.4, 68.3,
       68.5, 56.2, 65.9, 56.5, 56.1, 66.5, 68.4, 69.7, 57.1, 68.7, 56.7,
       68.6, 71.6, 43. , 68.8, 67.5, 69. , 55.2, 68.9, 69.6, 57. , 56.4,
       56.8, 44. , 67.2, 70.1, 71.3, 70.6, 69.8, 71

In [42]:
df['depth']=df['depth'].clip(50,70)
df['table']=df['table'].clip(50,70)

In [43]:
df.isnull().sum()

carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64

In [44]:
df.shape

(53940, 10)

In [45]:
df.duplicated().sum()

np.int64(146)

In [46]:
df.drop_duplicates(inplace=True)
df.duplicated().sum()

np.int64(0)

In [47]:
df['price'].skew()

np.float64(1.618239898265432)

In [48]:
import numpy as np
df['price_log']=np.log1p(df['price'])

Outliers in target variable → transform, not remove
Log transform is mainly for strong right skew
df[col].skew()>1

“I detected that the price variable was right-skewed with extreme high values.
Since those values are valid business cases, I didn’t remove them.
Instead, I applied a log transformation to stabilize variance and improve model performance

x=y=z=0 is an imposiible case so we remove the rows where x=y=z=0

In [49]:
df=df[(df['x']>0) & (df['x']>0) & (df['x']>0)]

In [50]:
df.shape# the reduced from 53940 to 53787

(53787, 11)

In [51]:
y=df['price_log']
x=df.drop(columns=['price','price_log'])

In [52]:
numeric=x.select_dtypes(include='number').columns.tolist()
categoric=x.select_dtypes(include='object').columns.tolist()

In [53]:
# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler
preprocessor=ColumnTransformer(transformers=[("cat",OrdinalEncoder(),categoric),("num",StandardScaler(),numeric)])

In [56]:
#pipeline
model_pipeline=Pipeline(steps=[('preprocessing',preprocessor),('model',KNeighborsRegressor())])

In [57]:
model_pipeline

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('cat', OrdinalEncoder(),
                                                  ['cut', 'color', 'clarity']),
                                                 ('num', StandardScaler(),
                                                  ['carat', 'depth', 'table',
                                                   'x', 'y', 'z'])])),
                ('model', KNeighborsRegressor())])

In [58]:
# Train-test split
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42)

In [59]:
model_pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('cat', OrdinalEncoder(),
                                                  ['cut', 'color', 'clarity']),
                                                 ('num', StandardScaler(),
                                                  ['carat', 'depth', 'table',
                                                   'x', 'y', 'z'])])),
                ('model', KNeighborsRegressor())])

In [60]:
y_pred=model_pipeline.predict(x_test)
print(y_test.shape)
print(y_pred.shape)

(13447,)
(13447,)


In [61]:
from sklearn.metrics import r2_score
score=r2_score(y_test,y_pred)
print(score)

0.9798862583857769


In [62]:
# save model
joblib.dump(model_pipeline,"model.pkl")

['model.pkl']

In [11]:
import sklearn
sklearn.__version__

'1.6.1'